In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import random
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors
import torch
import torch.nn as nn
import torch.optim as optim
import warnings
import math
warnings.filterwarnings('ignore')
from SurvivalEVAL.Evaluator import SurvivalEvaluator
from tqdm.auto import tqdm
import time
import os
import json
import torch
from sklearn.model_selection import StratifiedKFold
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print("="*80)
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("="*80)

--------

In [ ]:
def preprocess_df(df, duration_col, event_col, drop_cols=None):
    """Preprocess survival dataset"""
    df = df.copy()
    
    if drop_cols:
        df = df.drop(columns=drop_cols, errors='ignore')
    
    df = df[df[duration_col] > 0].copy()
    
    durations = df[duration_col].astype(float).values
    events = df[event_col].astype(int).values
    X = df.drop(columns=[duration_col, event_col])
    
    num_cols = X.select_dtypes(include=['number']).columns.tolist()
    cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()
    num_data = X[num_cols].copy().fillna(X[num_cols].median())
    
    if len(cat_cols) > 0:
        cat_data = pd.get_dummies(X[cat_cols].astype(str), drop_first=True)
        X_proc = pd.concat([num_data.reset_index(drop=True), cat_data.reset_index(drop=True)], axis=1)
    else:
        X_proc = num_data
    
    all_cols = X_proc.columns.tolist()
    return X_proc, durations, events, all_cols

# Models

------------

In [ ]:
from ensurv.model import EnSurv

# Evaluations w.r.t All Metrics

In [ ]:
def compute_all_metrics(model, y_tr, e_tr, X_te_scaled, y_te, e_te, y_max):
    """
    Computes metrics using SurvivalEVAL in the original unscaled time domain.
    """
    # The time grid spans the entire possible support so curves decay naturally.
    # Using the max of both sets guarantees we don't amputate the tail.
    global_max_time = max(y_tr.max(), y_te.max())
    
    # Bumped to 150 points to reduce linear interpolation artifacts in D-calibration
    times_grid = np.linspace(0.0, global_max_time, 100)
        
    # Scale the grid down to match the model's trained domain
    scaled_times_grid = times_grid / y_max
    
    # Query the model using the scaled grid to get accurate probabilities
    S_pred = model.predict_survival(X_te_scaled, scaled_times_grid, nsample=2000, batch_size=256, allow_extrapolation=True)
    
    # Feed the original unscaled data to the evaluator
    evl = SurvivalEvaluator(
        S_pred,
        times_grid,  # Original scale
        y_te,        # Original scale
        e_te,
        y_tr,        # Original scale
        e_tr,
        predict_time_method="Median"
    )
    
    d_cal_p, _ = evl.d_calibration()
    
    km_cal = evl.km_calibration()
    if isinstance(km_cal, tuple):
        km_cal = km_cal[0]
        
    median_time = np.median(y_te)

    def _get_val(metric_output):
        return metric_output[0] if isinstance(metric_output, tuple) else metric_output

    metrics = {
        "Harrell's C": _get_val(evl.concordance(method="Harrell")),
        "Antolini's C": _get_val(evl.concordance_time_dependent(method="Antolini")),
        "Hinge Error": _get_val(evl.mae(method="Hinge")),
        "Margin Error": _get_val(evl.mae(method="Margin")),
        "Uncensored MAE": _get_val(evl.mae(method="Uncensored")),
        "AUC": _get_val(evl.auc(target_time=median_time)),
        "Plain IBS": _get_val(evl.integrated_brier_score(IPCW_weighted=False)), 
        "AUPRC": _get_val(evl.auprc()),
        "KM-Calibration": km_cal
    }
    
    return metrics

In [ ]:
import gc
import os
import random
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from pysurvival.models.simulations import SimulationModel

if __name__ == "__main__":
    # Settings for the simulation
    sample_sizes = [5000, 10000, 20000]
    distributions = ['exponential', 'weibull', 'log-normal']
    risk_types = ['linear', 'square']
    censored_parameters = [5, 20, 35, 50]
    
    default_hyperparams = {
        "num_layer": 2,
        "hidden_dims": [64,64],
        "dropout": 0.1,
        "noise_dist": "gaussian",
        "noise_std": 1.0,
        "batch_normalization": False,
        "lr": 1e-3,
        "weight_decay": None
    }
    
    seed = 42
    n_folds = 3  
    num_features = 10
    export_path = "Ensurv_synthetic_simulation_results.csv"
    
    all_final_results = []

    # Calculate total iterations for a cleaner progress bar (Sizes * Dists * Risks * Censors)
    total_settings = len(sample_sizes) * len(distributions) * len(risk_types) * len(censored_parameters)
    pbar = tqdm(total=total_settings, desc="Overall Progress")

    for dist in distributions:
        for risk in risk_types:
            for cens_param in censored_parameters:
                for n in sample_sizes:
                    print(f"\n{'='*80}")
                    print(f"Processing: Dist={dist.title()} | Risk={risk.title()} | CensParam={cens_param} | N={n}")
                    print(f"{'='*80}")

                    if n == 5000 or n == 10000:
                        BATCH_SIZE = None
                    else:
                        BATCH_SIZE = 8888
                    
                    # Reproducibility setup
                    np.random.seed(seed)
                    random.seed(seed)

                    # Generate dataset using PySurvival
                    sim = SimulationModel(
                        survival_distribution=dist, 
                        risk_type=risk,
                        censored_parameter=cens_param,  
                        alpha=0.1,               
                        beta=2.0                 
                    )
                    
                    dataset = sim.generate_data(
                        num_samples=n, 
                        num_features=num_features
                    )
                    
                    # Extract features, times, and events
                    durations = np.clip(dataset['time'].values, a_min=1e-5, a_max=None)
                    events = dataset['event'].values
                    X = dataset.drop(['time', 'event'], axis=1)
                    
                    
                    # Calculate actual empirical censoring rate
                    actual_censoring_rate = (1.0 - events.mean()) * 100
                    print(f"  -> Actual generated censoring rate: {actual_censoring_rate:.2f}%")

                    if actual_censoring_rate < 10.0:
                        print("  -> Skipping configuration (censoring rate < 10%)")
                        pbar.update(1)  # Ensure progress bar advances before skipping
                        continue
                    
                    # K-fold setup
                    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
                    splits = list(kf.split(X, events))
                    
                    metrics_storage = {}

                    for fold_offset, (train_idx, test_idx) in enumerate(splits):
                        actual_fold = fold_offset + 1
                        print(f"  -> Training & Evaluating Fold {actual_fold}/{n_folds}")
                        
                        X_tr, y_tr, e_tr = X.iloc[train_idx].values, durations[train_idx], events[train_idx]
                        X_te, y_te, e_te = X.iloc[test_idx].values, durations[test_idx], events[test_idx]
                        
                        scaler = StandardScaler()
                        X_tr_scaled = scaler.fit_transform(X_tr).astype(np.float32)
                        X_te_scaled = scaler.transform(X_te).astype(np.float32)
                        
                        
                        # Extract scaling factor based strictly on training data
                        y_max = y_tr.max()
                        y_tr_scaled = y_tr / y_max
                        y_tr_scaled = (y_tr / y_max).astype(np.float32)
                        e_tr = e_tr.astype(np.float32)
                        
                        model = EnSurv(
                            latent_dim=None,
                            num_layer=default_hyperparams["num_layer"],
                            hidden_dims=default_hyperparams["hidden_dims"],
                            dropout=default_hyperparams["dropout"],
                            noise_dist=default_hyperparams["noise_dist"],
                            noise_std=default_hyperparams['noise_std'], 
                            batch_normalization=default_hyperparams['batch_normalization'],
                            device=DEVICE,
                            random_state=seed, generator='concatenated'
                        )
                        
                        # Fit the model
                        model.fit(
                            X_tr_scaled, y_tr_scaled, e_tr, 
                            epochs=200,
                            m=10, 
                            B=20, 
                            verbose=True,
                            patience=15, 
                            weight_decay=default_hyperparams["weight_decay"], track_diagnostics=False,
                            lambda_energy=0.5, batch_size=BATCH_SIZE, lr=default_hyperparams["lr"], km_batch_size=256
                        )
                        
                        # Pass original arrays + y_max for proper unscaled evaluation
                        fold_metrics = compute_all_metrics(model, y_tr, e_tr, X_te_scaled, y_te, e_te, y_max)
                        
                        for metric_name, val in fold_metrics.items():
                            if metric_name not in metrics_storage:
                                metrics_storage[metric_name] = []
                            metrics_storage[metric_name].append(val)
                        
                        # GPU Memory Cleanup
                        del model
                        del X_tr, y_tr, e_tr, X_te, y_te, e_te, y_tr_scaled, X_tr_scaled, X_te_scaled
                        gc.collect()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()

                    # 4. Aggregation & Formatting
                    for metric_name, values in metrics_storage.items():
                        mean_val = np.nanmean(values)
                        std_val = np.nanstd(values)
                        
                        formatted_result = rf"${mean_val:.4f}\pm{std_val:.4f}$"
                        
                        all_final_results.append({
                            "Distribution": dist.title(),
                            "Risk Type": risk.title(),
                            "Censored Param": cens_param,
                            "Actual Censoring": f"{(actual_censoring_rate):.2f}%",
                            "# Samples": n,
                            "Metric": metric_name,
                            "EnSurv": formatted_result
                        })

                    # 5. Continuous Overwrite/Export
                    df_final = pd.DataFrame(all_final_results)
                    df_final = df_final[["Distribution", "Risk Type", "Censored Param", "Actual Censoring", "# Samples", "Metric", "EnSurv"]] 
                    df_final.to_csv(export_path, index=False)
                    print(df_final.to_string(index=False), flush=True)
                    
                    pbar.update(1)

    pbar.close()
    
    print(f"\nAll simulations complete. Final results cleanly saved to {export_path}.")
    display(df_final)